## Загрузка датасета на HuggingFace

Этот ноутбук дает пример того, как залить локальный датасет на ХФ. Адаптируйте его под свой датасет. Затем, выложите в гитхаб получившийся ноутбук (приложите к своему датасету), чтобы всегда был доступен код для заливки вашего датасета на ХФ. Убедитесь, что ячейки последовательно запускаются.

In [1]:
from PIL import Image
import json
import datasets
from tqdm import tqdm
import os

/home/jovyan/.mlspace/envs/akharitonov_rusregions_pr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Подготовка данных

#### WARNING! 

Если ваш датасет является __ПРИВАТНЫМ__, то оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`. Иначе, поставьте `False`. Этот флаг дальше используется, чтобы стереть ответы перед загрузкой на ХФ датасета. На ХФ даже приватно не должно лежать датасетов с ответами!

In [2]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True

Данный пример рассчитан на загрузку на ХФ локального датасета.

Параметр `path_to_data` - это путь ДО файлов `shots.json` и `test.json`, которые вы будете дальше загружать на ХФ в виде датасета или домена датасета. 

Параметр `path_to_meta` - это путь ДО меты датасета.

Итоговые пути будут собираться из `path_to_data` / `path_to_meta` + `file_name.json`!

In [3]:
path_to_data = "./"
path_to_meta = "./"

Сплиты и мета лежат в формате JSON.

In [4]:
def load_json(path):
    with open(path) as f:
        data = json.load(f)
    return data

#### Подгрузка данных

Считайте сплиты и мету датасета (домена датасета). Это просто JSON файлики либо внутри прямо папки датасета, либо внутри папки по названию домена, который вы будете загружать.

In [5]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

Из меты для датасета нужны только промпты.

In [ ]:
# prompts = meta["prompts"]

#### Обработка полей датасета

На ХФ вы загружаете датасет, где у КАЖДОГО сэмпла вместо числа в поле instruction стоит промпт. Число указывает, какой по индексу взять промпт из секции с промптами в мете датасета.

In [ ]:
# for card in shots:
#     card["instruction"] = prompts[card["instruction"]]

# for card in test:
#     card["instruction"] = prompts[card["instruction"]]

#### Убираем ответы для приватных задач

Надеемся, вы поставили в начале ноутбука корректное значение `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS`.

Если там стоит `True`, то в `test` сплите ответы на все задания стираются. Вместо них остается пустая строка, чтобы вы случайно не пушнули на ХФ датасет с заполненными ответами, и они не утекли.

In [6]:
def hide_answers(dataset_split: list[dict]):
    for card in tqdm(dataset_split):
        card["outputs"] = ""

In [7]:
if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    hide_answers(test)

100%|██████████| 603/603 [00:00<00:00, 3413178.56it/s]


### Создаем датасет для загрузки на ХФ

#### Аннотация полей датасета

В `features` повторяется структура КАЖДОГО сэмпла вашего датасета с описанием формата данных в каждом поле. 
- `instruction` всегда строка
- `meta` - id всегда целое число

Далее смотрите по тому, какие поля у вашего датасета.

`features` нужен для того, чтобы ХФ сам автоматически создал техническую часть README.md датасета, заполнив ее информацией, которая используется при загрузке датасета. Отсутствие `features` может и обычно приводит к невозможности использовать датасет. Ровно такие же последствия будут от ошибок в заполнении (например, неправильно указан тип данных).

__Внимание!__ Если у вас в датасете в разных вопросах разное количество ответов, то поля в `features` нужно заполнить для сэмпла с НАИБОЛЬШИМ количеством ответов. Иначе говоря, представьте, что у вас у всех вопросов в датасете максимальное количество вариантов ответа, просто некоторые пустые. Вот из такого соображения и заполняйте `features`. Он один на весь датасет и должен охватывать все поля, которые в нем встречаются!

In [8]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "option_a": datasets.Value("string"),
        "option_b": datasets.Value("string"),
        "option_c": datasets.Value("string"),
        "option_d": datasets.Value("string"),
        "condition": datasets.Value("string"),
        "task_formulation": datasets.Value("string"),
        "format_description": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "group_id": datasets.Value("int32"),
        "task_part": datasets.Value("int32"),
    },
})


#### Создание датасетов для каждого сплита

Теперь создаем сплиты датасета. Можно это сделать либо в одну строку:

In [9]:
shots_ds = datasets.Dataset.from_list(shots, features=features)

Но это способ для маленьких датасетов. Большие датасеты так создаются крайне долго. Чтобы побыстрее собрать большой датасет, можно разбить его на кусочки по N сэмплов. Перегонять каждый кусочек и присоединять к уже конвертированным ранее кусочкам.

In [10]:
STEP = 20

lst_steps = []
for i in tqdm(range(0, len(test), STEP)):
    tmp = datasets.Dataset.from_list(test[i: i+STEP], features=features)
    lst_steps.extend([tmp])
    
test_ds = datasets.concatenate_datasets(lst_steps)

100%|██████████| 31/31 [00:00<00:00, 93.01it/s]


In [11]:
shots_ds = shots_ds.cast(features)
test_ds = test_ds.cast(features)

Casting the dataset: 100%|██████████| 603/603 [00:00<00:00, 112212.84 examples/s]


##### Проверка

Если вы собирали датасет по кускам, то разумно будет проверить, что сборка прошла успешно - ничего не потеряно, не продублировано и так далее.

Но вы можете проверить целостность датасета даже, если и не по кусочкам собирали его. Так вы можете отловить ошибки до того, как их найдут на ревью :)

In [12]:
# проверка, что id вопросов сходятся

bools = []
for i in range(len(test)):
    bools.extend([test[i]["meta"]["id"] == test_ds[i]["meta"]["id"]])
all(bools)

True

In [13]:
# проверка, что количество вопросов до конвертации и после осталось одинаковым

len(test) == len(test_ds)

True

#### Собираем сплиты в один датасет

In [14]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})

### Загрузка датасета на ХФ

Для загрузки на ХФ вам понадобятся:
- Токен. Это строка, содержащая ключик, который позволит вам записывать в репозиторий. 
- Путь для записи. Это тоже строка, которая содержит путь, по которому вы выложите свой датасет. Этот путь содержит название аккаунта (MERA-evaluation) и название вашего датасета. Название датасета пишите ровно так, как оно заявлено в мете! Регистр тоже имеет значение!

Советуем опубликовывать сперва всё приватно, и выслать на почту mera@a-ai.ru токен и путь для верификации. 
Если ваш сет публичный и вы хотите отправить всё публично, то в Merge request просто пришлите путь к сету.

In [ ]:
### TOKEN
token = ""
###

### UPLOAD PATH
dataset_path_hub = "MERA-evaluation/RuRegions"
###


# Если вы хотите предварительно протестировать, как датасет будет выглядеть после заливки на ХФ,
# то можно загрузить его сначала к себе в приватный репозиторий

### UPLOAD PATH
# dataset_path_hub = "artemorloff/ruclevr"
###

In [16]:
dataset.push_to_hub(dataset_path_hub, private=True, token=token) # опубликовать приватно

Setting num_proc from 1 back to 1 for the shots split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 322.44ba/s]
Processing Files (1 / 1): 100%|██████████| 31.0kB / 31.0kB, 17.2kB/s  
New Data Upload: 100%|██████████| 31.0kB / 31.0kB, 17.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.95s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 308.43ba/s]
Processing Files (1 / 1): 100%|██████████| 1.10MB / 1.10MB,  788kB/s  
New Data Upload: 100%|██████████| 1.10MB / 1.10MB,  788kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.13s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/MERA-evaluation/RuRegions/commit/24300dae242756832448a62d51396fcd1b194ed7', commit_message='Upload dataset', commit_description='', oid='24300dae242756832448a62d51396fcd1b194ed7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MERA-evaluation/RuRegions', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MERA-evaluation/RuRegions'), pr_revision=None, pr_num=None)

### Проверка того, как датасет загрузился на ХФ

После загрузки датасета будет полезно посмотреть, как его будет видеть любой человек, который после вашей загрузки его скачает. 

Загрузите датасет целиком, используя `datasets.load_dataset(dataset_path_hub)`, а затем проверьте, что:
- все поля на месте. Если у вас в датасете у разных вопросов было разное количество вариантов ответа, то теперь их везде станет одинаковое количество. Недостающие варианты ответа у каждого вопроса теперь будут прописаны, но будут иметь значение `None`. Это нормально.
- ваша модальность корректно обработалась. Если у вас в датасете были картинки, то все они должны превратиться в байткод. Не должно остаться ни одной картинки, которая не конвертирована в байты. Если у картинки есть и байты, и путь прописан (а не `None`), то это окей. `bytes` точно должны быть заполнены, `path` может быть None.
- датасет идентичен по содержанию исходному. То есть, в исходном JSON и загруженном датасете вопрос с одинаковым `id` имеет одинаково заполненные поля (кроме тех, что заполняются `None`, как описано выше).

In [18]:
ds = datasets.load_dataset(dataset_path_hub, token=token)

Generating test split: 100%|██████████| 603/603 [00:00<00:00, 34248.72 examples/s]


In [19]:
ds

DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 9
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 603
    })
})

In [23]:
ds["test"][0]

{'instruction': '{condition}\nА. {option_a}\nБ. {option_b}\nВ. {option_c}\nГ. {option_d}\n\n{task_formulation}\n\n{format_description}',
 'inputs': {'option_a': 'Прогуливаясь по проспекту, обратил внимание на обилие жидрика, оккупировавшего все свободные лавочки и бордюры в ожидании добродушных прохожих.',
  'option_b': 'Прежде чем выдавать что-то за свою жмычку, хорошо бы узнать, есть ли уже у кого-нибудь на неё патент. Иначе можно проколоться.',
  'option_c': 'Поутру всё поле было окутано куржаком! От увиденного захватывало дух, даже не верилось, что всё это было создано природой. Всё-таки великолепно! Ничего не скажешь.',
  'option_d': 'Закрыв пастик и собрав весь свой немногочисленный скарб, он вышел из дома и тут же был с ног до головы облит грязью от проезжавшего мимо аптрагана.',
  'condition': 'Дан следующий перечень текстов:',
  'task_formulation': 'Произведите выбор того элемента из представленных, который является семантически корректным.',
  'format_description': 'Результат

Пример проверки двух сплитов, что в них тексты вопросов совпадают с оригинальными

In [21]:
check = []
for idx, card in enumerate(ds["shots"]):
    same_question = shots[idx]["inputs"]["task_formulation"] == card["inputs"]["task_formulation"]
    check.extend([same_question])

all(check)

True

In [22]:
check = []
for idx, card in enumerate(ds["test"]):
    same_question = test[idx]["inputs"]["task_formulation"] == card["inputs"]["task_formulation"]
    check.extend([same_question])

all(check)

True